# 3. MERGE y REINSPECCION FINAL

In [10]:
import pandas as pd
from pathlib import Path


ROOT = Path.cwd().resolve()
if ROOT.name == "jupyter":
    ROOT = ROOT.parent
CLEAN = ROOT / "CLEAN"
DATASET_EDA = ROOT / "DATASETS_EDA"

CLEAN.mkdir(parents=True, exist_ok=True)

CSV_FILES = ["descubrimientos.csv","estrellas.csv","mediciones.csv","planetas.csv","sistemas.csv"]

d = pd.read_csv(CLEAN / "descubrimientos.csv")
e = pd.read_csv(CLEAN / "estrellas.csv")
m = pd.read_csv(CLEAN / "mediciones.csv")
p = pd.read_csv(CLEAN / "planetas.csv")
s = pd.read_csv(CLEAN / "sistemas.csv")

Antes del merge de los diferentes dataframes se filtra por flag = 1, 
la cual apunta a la ultima fecha de actualizacion de mediciones, se dropean las demas filas,
creando un df intermedio y guardandolo como csv para su posterior merge.

In [11]:
print(m.head())
mask_flag = m["default_flag"] == 1
m2 = m.loc[mask_flag]
print(m2.head)
print(m2.shape)

m2.to_csv(CLEAN / "mediciones2.csv", index=False)


  measurement_id   pl_name hostname  default_flag  pl_orbper  pl_rade  \
0      MED000001  11 Com b   11 Com             0        NaN      NaN   
1      MED000002  11 Com b   11 Com             1     323.21      NaN   
2      MED000003  11 Com b   11 Com             0     326.03      NaN   
3      MED000004  11 UMi b   11 UMi             0     516.22      NaN   
4      MED000005  11 UMi b   11 UMi             0        NaN      NaN   

   pl_orbeccen  pl_eqt  st_teff  st_mass pl_pubdate releasedate  
0          NaN     NaN      NaN     2.60    2011-08  2014-07-23  
1        0.238     NaN   4874.0     2.09    2023-12  2023-09-19  
2        0.231     NaN   4742.0     2.70    2008-01  2014-05-14  
3        0.080     NaN   4340.0     1.80    2009-10  2014-05-14  
4          NaN     NaN      NaN     1.70    2011-08  2014-07-23  
<bound method NDFrame.head of       measurement_id     pl_name  hostname  default_flag    pl_orbper  \
1          MED000002    11 Com b    11 Com             1   323

Se inspeccioann duplicados de nuevo

In [12]:
print(d.duplicated().sum())
print(e.duplicated().sum())
print(m2.duplicated().sum())
print(p.duplicated().sum())
print(s.duplicated().sum())

0
0
0
0
0


Merge y guardado del dataset final

In [13]:
dataset = m2.merge(
    p,
    on="pl_name",
    how="left"
).sort_values(by="pl_name", ascending=False)
print(dataset.shape)

dataset = dataset.merge(
    d,
    on="pl_name",
    how="left"
).sort_values(by="pl_name", ascending=False)
print(dataset.shape)

dataset = dataset.merge(
    e,
    on="hostname",
    how="left"
).sort_values(by="hostname", ascending=False)
print(dataset.shape)

dataset = dataset.merge(
    s,
    on="hostname",
    how="left"
).sort_values(by="hostname", ascending=False)
print(dataset.shape)

dataset.to_csv(CLEAN / "dataset_merged.csv", index=False)

(6464, 22)
(6464, 29)
(6464, 36)
(6464, 41)


Se dropean columnas que no se han conseguido matchear del todo, despues de inspeccion

In [14]:
df = pd.read_csv(CLEAN/"dataset_merged.csv")

In [15]:
print(df.head)
print(df.columns)
print(df.shape)
print(df.info())
df= df.drop(columns=["hostname_y","pl_orbper_y","pl_orbeccen_y","pl_eqt_y","pl_pubdate_y",
    "releasedate_y","st_teff_y","st_mass_y"])
print(df.shape)

<bound method NDFrame.head of      measurement_id      pl_name hostname_x  default_flag  pl_orbper_x  \
0         MED040386     xi Aql b     xi Aql             1   136.970000   
1         MED040384    ups Leo b    ups Leo             1   385.200000   
2         MED040382    ups And d    ups And             1  1276.460000   
3         MED040374    ups And c    ups And             1   241.258000   
4         MED040363    ups And b    ups And             1     4.617033   
...             ...          ...        ...           ...          ...   
6459      MED002469  HD 145675 c     14 Her             1          NaN   
6460      MED000014     14 Her b     14 Her             1          NaN   
6461      MED000007     14 And b     14 And             1   186.760000   
6462      MED000006     11 UMi b     11 UMi             1   516.219970   
6463      MED000002     11 Com b     11 Com             1   323.210000   

      pl_rade_x  pl_orbeccen_x  pl_eqt_x  st_teff_x  st_mass_x  ... st_rad  \
0  